# Aprendizado por Reforço

---

**Professor:** Prof. Gabriel Lima  
**Aula:** 04  
**Exercício:** 4B  

---

### Objetivo :  
Para o problema randon Walk implementar os algoritmos descritos abaixo: 

- First Visit Monte Carlo Policy Evaluation
- TD(0) Policy Evaluation

## Bibliotecas Utilizadas

Para a implementação dos algoritmos utilizaremos as seguintes bibliotecas: 

In [1]:
import numpy as np

## Definição do Ambiente

Considere o seguinte ambiente a ser implementado: 

<img src="RandonWalk.png" width="300"/>

O agente começa no estado C e recebe recompensas 

ℛ(𝑠,𝑎)=0,∀𝑠 ∈𝒮= {𝐴,𝐵,𝐶,𝐷,𝐸}, ∀𝑎 ∈𝒜={←,→}, exceto para ℛ(𝐸,→)=1

In [2]:
class RandomWalkEnv:

  def __init__(self,NumberOfStates:int,gamma:float,initialStateProbs:list):
    self.NumberOfStates    = NumberOfStates
    self.gamma             = gamma
    self.initialStateProbs = initialStateProbs
    self.StateSpace        = list(np.arange(NumberOfStates))
    self.actionSpace       = [-1,1]


  def step(self, state, action):

    # state(0)
    if (state == 0 and action == -1):
      reward = 0
      NextState = -1
      done = True

    # state(NumberofStates)
    elif (state == (self.NumberOfStates -1) and action == 1):
      reward = 1
      NextState = self.NumberOfStates
      done = True

    else:
      reward = 0
      NextState = state + action
      done = False

    return NextState,reward,done


  def show(self,state):
    Map = np.zeros(self.NumberOfStates)
    Map[state] = 1
    return Map

## Definição da Classe de Transição

In [3]:
class Transition():
    def __init__(self,state,action,reward,next_state,done):
        self.state = state
        self.action = action
        self.reward = reward
        self.next_state = next_state
        self.done = done

## Instanciando as Classes

In [4]:
env = RandomWalkEnv(NumberOfStates=5, gamma=1, initialStateProbs=[0,0,1,0,0])

In [5]:
n = len(env.StateSpace)
m = len(env.actionSpace)

In [6]:
print(n)
print(m)

5
2


## Definindo a Política

Adimitindo uma política equiprobabilistica:

In [7]:
random_policy = (1/m)*np.ones((n,m))

# Mude para uma política enviesada !
#random_policy = np.tile([0.2, 0.8], (n, 1))

In [8]:
random_policy

array([[0.5, 0.5],
       [0.5, 0.5],
       [0.5, 0.5],
       [0.5, 0.5],
       [0.5, 0.5]])

## Função Episódio

In [9]:
def episodes(env,policy):
  ep = []
  done  = False
  state = np.random.choice(env.StateSpace, p =env.initialStateProbs)

  while not done:
    action = np.random.choice(env.actionSpace,p=policy[state,:])
    next_state, reward, done = env.step(state,action)

    transition = Transition(state,action,reward,next_state,done)
    ep.append(transition)
    state = next_state

  return ep

## First Visit

Verifica se um determinado estado (state) está aparecendo pela primeira vez em uma lista de estados já visitados (visited_states).

In [10]:
def first_visit(visited_states, state):
    first_visit = True
    for s in visited_states:
        if np.all(s == state):
            first_visit = False

    return first_visit

## First Visit Monte Carlo Policy Evaluation

### Algoritmo

**Entrada:**
- Política 𝜋  
- Número de episódios 𝑁  
- Environment

**Inicialização:**
- V(s) ← 0, ∀ s ∈ 𝒮  
- N(s) ← 0 (contador de visitas), ∀ s ∈ 𝒮

**Repetir para cada episódio (até N episódios):**  
1. Gerar um episódio completo seguindo a política 𝜋:  
   → sequência: (s₀, a₀, r₁, s₁, a₁, r₂, ..., s_T)  
2. G ← 0  
3. visited ← []  
4. Para t = T − 1 até 0 (de trás pra frente):  
   a. G ← γ × G + rₜ₊₁  
   b. Se sₜ **ainda não apareceu** em `visited`:  
      i. adicionar sₜ à lista `visited`  
      ii. N(sₜ) ← N(sₜ) + 1  
      iii. V(sₜ) ← V(sₜ) + (1 / N(sₜ)) × [G − V(sₜ)]

**Saída:**  
- Função de valor V(s) para todos os estados sob a política 𝜋

In [11]:
def first_visit_monte_carlo_policy_evaluation(env,policy,N_eps=1000):
    n = len(env.StateSpace)
    # Initialize Sum of Returns S and State Visit Count N
    S = np.zeros(n)
    N = np.zeros(n)
    # Loop over episodes
    for i in range(N_eps):
        episode = episodes(env,policy)
        G = 0
        T = len(episode)
        # Loop over timesteps on episode
        for t in range(T-1,-1,-1):

            transition = episode[t]
            G = env.gamma*G + transition.reward
            visited_states = [e.state for e in episode[0:t]]
            if (first_visit(visited_states, transition.state)):
              N[transition.state] += 1
              S[transition.state] += G

    # Value Function Estimate V ~ S/N

    V = S/N
    return V

In [12]:
V = first_visit_monte_carlo_policy_evaluation(env,random_policy,N_eps=20000)

In [13]:
V

array([0.17103391, 0.33783603, 0.5049    , 0.6719457 , 0.83551216])

## Incremental First Visit Monte Carlo Policy Evaluation

### Algoritmo:

**Entrada:**
- Política 𝜋  
- Número de episódios 𝑁  
- Environment
- Taxa de aprendizado α ∈ (0, 1]  

**Inicialização:**
- V(s) ← 0, ∀ s ∈ 𝒮  
- N(s) ← 0, ∀ s ∈ 𝒮  (contador de visitas)

**Para cada episódio (até N episódios):**
1. Gerar um episódio completo seguindo a política 𝜋:  
   → sequência: (s₀, a₀, r₁, s₁, a₁, r₂, ..., s_T)  
2. G ← 0  
3. visited ← []  
4. Para t = T − 1 até 0 (de trás pra frente):  
   a. G ← γ × G + rₜ₊₁  
   b. Se sₜ **não está em** `visited`:  
      i. visited.append(sₜ)  
      ii. V(sₜ) ← V(sₜ) + α × [G − V(sₜ)]

**Saída:**
- Função de valor V(s) para todos os estados sob a política 𝜋


In [14]:
def Incremental_First_Visit_Monte_Carlo_Evaluation(env,policy,alpha,N_eps):
  Vk = np.zeros(env.NumberOfStates)
  for i in range(N_eps):
    Vk1 = np.copy(Vk)
    episode  = episodes(env,policy)
    G = 0
    T = len(episode)
    returns_list = []
    for t in range(T-1,-1,-1):
        transition = episode[t]
        G = env.gamma*G + transition.reward
        returns_list.append(G)
    returns_list = list(reversed(returns_list))

    Already_Visited = [False for i in range(env.NumberOfStates)]

    for t in range(T):
      transition = episode[t]
      state = transition.state
      G = returns_list[t]

      if (Already_Visited[state] == False):
        Already_Visited[state] =  True
        Vk1[state] = Vk[state] + alpha*(G-Vk[state])

    Vk = Vk1

  return Vk1

In [15]:
V_alpha = Incremental_First_Visit_Monte_Carlo_Evaluation(env,random_policy,alpha=0.01,N_eps=20000)

In [16]:
V_alpha

array([0.24555495, 0.39877092, 0.54965313, 0.6901678 , 0.85001729])

## TD(0) Policy Evaluation

### Algoritmo:

**Entrada:**
- Número de episódios 𝑁  
- Taxa de aprendizado α ∈ (0, 1]  
- Environment

**Inicialização:**
- V(s) ← 0, ∀ s ∈ 𝒮

**Repetir para cada episódio (até N episódios):**  
1. Inicializar o estado s ← estado inicial do episódio  
2. Enquanto s não for terminal:  
   a. Escolher ação a ← 𝜋(s)  
   b. Executar a ação a, observar recompensa r e próximo estado s′  
   c. Atualizar:  
      V(s) ← V(s) + α × [r + γ × V(s′) − V(s)]  
   d. Atualizar o estado atual: s ← s′

**Saída:**  
- Função de valor V(s) para todos os estados sob a política 𝜋

In [17]:
def TD0_Policy_Evaluation(env,alpha,N_eps):
  V = np.zeros(env.NumberOfStates)

  for ep in range(N_eps):
    episode = episodes(env,random_policy)
    T = len(episode)

    for t in range(T):
      state = episode[t].state
      next_state = episode[t].next_state
      done = episode[t].done
      reward = episode[t].reward

      if not done:
        V[state] = V[state] + alpha *(reward + (env.gamma*V[next_state]) \
                                                             - V[state])
      else:

        V[state] = V[state] + alpha *(reward + (env.gamma*0 - V[state]))

  return V

In [18]:
V_td0 = TD0_Policy_Evaluation(env,alpha=0.05,N_eps=20000)

In [19]:
V_td0

array([0.12193778, 0.27720762, 0.52027161, 0.71918619, 0.8706586 ])